# Stock Bot Walk-Forward
Runs historical validation only. It does not use Alpaca credentials or publish a live config.

In [ ]:
# Mount Drive so snapshots and checkpoints survive a Colab disconnect.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Change SNAPSHOT_NAME to the file created on your computer.
REPO_URL = 'https://github.com/uwuexdeemeow/Stock-Market-AI-Bot.git'
SNAPSHOT_NAME = 'CHANGE_ME.tar.gz'
DRIVE_DIR = '/content/drive/MyDrive/StockBotWalkforward'
GRID_FLAG = '--low-turnover-grid'
OUTPUT_PREFIX = 'wf_colab_reliability'


In [ ]:
# Clone the exact code version and verify the uploaded snapshot checksum.
import hashlib, json, os, pathlib, shutil, subprocess, tarfile
snapshot = pathlib.Path(DRIVE_DIR) / SNAPSHOT_NAME
manifest = snapshot.with_suffix('.manifest.json')
info = json.loads(manifest.read_text())
digest = hashlib.sha256(snapshot.read_bytes()).hexdigest()
assert digest == info['sha256'], 'Snapshot checksum does not match'
repo = pathlib.Path('/content/stockbot')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', 'checkout', info['git_commit']], cwd=repo, check=True)
with tarfile.open(snapshot, 'r:gz') as handle: handle.extractall(repo)
print('Snapshot verified:', info['file_count'], 'files')


In [ ]:
# Install the same project dependencies used locally.
subprocess.run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=repo, check=True)
subprocess.run(['python3', 'factor_data_health.py', '--strict', '--no-write'], cwd=repo, check=True)


In [ ]:
# Run two outer folds per process. Copy checkpoints to Drive after each batch.
import multiprocessing
checkpoint = repo / 'signals/walkforward_checkpoint_core_alpha.json'
drive_checkpoint = pathlib.Path(DRIVE_DIR) / checkpoint.name
if drive_checkpoint.exists(): shutil.copy2(drive_checkpoint, checkpoint)
workers = max(1, min(8, multiprocessing.cpu_count()))
final_json = repo / f'signals/{OUTPUT_PREFIX}.json'
for batch in range(10):
    cmd = ['python3', 'core_satellite_nested_walkforward.py', '--strategy', 'core-alpha', GRID_FLAG, '--resume', '--exit-after-folds', '2', '--workers', str(workers), '--output-prefix', OUTPUT_PREFIX, '--no-publish-live-config']
    subprocess.run(cmd, cwd=repo, check=True)
    if checkpoint.exists(): shutil.copy2(checkpoint, drive_checkpoint)
    if final_json.exists(): break
assert final_json.exists(), 'Walk-forward did not finish within 10 batches'
print('Walk-forward complete:', final_json)


In [ ]:
# Analyze and package the result for review on the project computer.
csv_path = repo / f'signals/{OUTPUT_PREFIX}.csv'
subprocess.run(['python3', 'walkforward_analyzer.py', '--csv', str(csv_path), '--json'], cwd=repo, check=True)
result_archive = pathlib.Path(DRIVE_DIR) / 'stockbot_colab_result.tar.gz'
with tarfile.open(result_archive, 'w:gz') as handle:
    for path in [final_json, csv_path, csv_path.with_suffix('.analyzer.json'), repo / 'signals/research_run_manifest.json']:
        if path.exists(): handle.add(path, arcname=path.relative_to(repo))
print('Saved result:', result_archive)
